# Analisis Struktur Data Excel dan Pembuatan Skema MySQL

Notebook ini akan membaca struktur data dari file Excel `data_latih.xlsx` dan `data_uji_y.xlsx`, kemudian membuat skema SQL MySQL berdasarkan struktur data tersebut.

## 1. Import Required Libraries

Import pandas untuk membaca file Excel dan library lain yang diperlukan.

In [ ]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

print("Libraries imported successfully!")

## 2. Load Excel Files

Memuat file Excel `data_latih.xlsx` dan `data_uji_y.xlsx` ke dalam pandas DataFrame.

In [ ]:
# Load data_latih.xlsx
try:
    df_latih = pd.read_excel('data_latih.xlsx')
    print(f"✓ data_latih.xlsx loaded successfully! Shape: {df_latih.shape}")
except Exception as e:
    print(f"❌ Error loading data_latih.xlsx: {e}")

# Load data_uji_y.xlsx
try:
    df_uji_y = pd.read_excel('data_uji_y.xlsx')
    print(f"✓ data_uji_y.xlsx loaded successfully! Shape: {df_uji_y.shape}")
except Exception as e:
    print(f"❌ Error loading data_uji_y.xlsx: {e}")

## 3. Inspect Data Structure

Memeriksa kolom, tipe data, dan contoh data dari kedua DataFrame untuk menentukan tipe kolom SQL yang sesuai.

In [ ]:
print("=" * 60)
print("ANALISIS STRUKTUR DATA_LATIH.XLSX")
print("=" * 60)

if 'df_latih' in globals():
    print(f"\nShape: {df_latih.shape}")
    print(f"\nColumn Names:")
    for i, col in enumerate(df_latih.columns):
        print(f"  {i+1}. {col}")
    
    print(f"\nData Types:")
    print(df_latih.dtypes)
    
    print(f"\nFirst 5 rows:")
    display(df_latih.head())
    
    print(f"\nInfo:")
    df_latih.info()
    
    print(f"\nNull values:")
    print(df_latih.isnull().sum())
else:
    print("❌ df_latih not loaded")

In [ ]:
print("=" * 60)
print("ANALISIS STRUKTUR DATA_UJI_Y.XLSX")
print("=" * 60)

if 'df_uji_y' in globals():
    print(f"\nShape: {df_uji_y.shape}")
    print(f"\nColumn Names:")
    for i, col in enumerate(df_uji_y.columns):
        print(f"  {i+1}. {col}")
    
    print(f"\nData Types:")
    print(df_uji_y.dtypes)
    
    print(f"\nFirst 5 rows:")
    display(df_uji_y.head())
    
    print(f"\nInfo:")
    df_uji_y.info()
    
    print(f"\nNull values:")
    print(df_uji_y.isnull().sum())
else:
    print("❌ df_uji_y not loaded")

## 4. Generate MySQL CREATE TABLE Statements

Membuat pernyataan CREATE TABLE MySQL secara otomatis berdasarkan skema yang diinferkan dari kedua DataFrame.

In [ ]:
def pandas_to_mysql_type(dtype, column_name, sample_data=None):
    """
    Convert pandas dtype to MySQL data type
    """
    dtype_str = str(dtype).lower()
    
    # Check for specific column patterns first
    if 'id' in column_name.lower() and ('int' in dtype_str or 'object' in dtype_str):
        return 'INT AUTO_INCREMENT PRIMARY KEY'
    
    # Map pandas dtypes to MySQL types
    if 'int' in dtype_str:
        if 'int64' in dtype_str:
            return 'BIGINT'
        else:
            return 'INT'
    elif 'float' in dtype_str:
        return 'DECIMAL(10,2)'
    elif 'bool' in dtype_str:
        return 'BOOLEAN'
    elif 'datetime' in dtype_str:
        return 'DATETIME'
    elif 'object' in dtype_str:
        # For object type, we need to check the actual data
        if sample_data is not None:
            max_length = 0
            for val in sample_data.dropna():
                if isinstance(val, str):
                    max_length = max(max_length, len(str(val)))
            
            if max_length <= 50:
                return 'VARCHAR(50)'
            elif max_length <= 255:
                return 'VARCHAR(255)'
            else:
                return 'TEXT'
        else:
            return 'VARCHAR(255)'
    else:
        return 'TEXT'

def generate_create_table_sql(df, table_name):
    """
    Generate CREATE TABLE SQL statement from pandas DataFrame
    """
    sql = f"CREATE TABLE {table_name} (\n"
    
    columns = []
    for column in df.columns:
        # Clean column name (remove spaces, special chars)
        clean_column = column.replace(' ', '_').replace('-', '_').replace('(', '').replace(')', '')
        clean_column = ''.join(c for c in clean_column if c.isalnum() or c == '_')
        
        # Get MySQL type
        mysql_type = pandas_to_mysql_type(df[column].dtype, clean_column, df[column])
        
        # Add NOT NULL constraint for non-nullable columns (except for AUTO_INCREMENT)
        if df[column].isnull().sum() == 0 and 'AUTO_INCREMENT' not in mysql_type:
            mysql_type += ' NOT NULL'
        
        columns.append(f"    {clean_column} {mysql_type}")
    
    sql += ",\n".join(columns)
    sql += "\n);\n"
    
    return sql

print("Functions for MySQL schema generation created successfully!")

In [ ]:
# Generate CREATE TABLE for data_latih
print("=" * 60)
print("GENERATING MySQL CREATE TABLE STATEMENTS")
print("=" * 60)

create_statements = []

if 'df_latih' in globals():
    print("\n1. CREATE TABLE for data_latih:")
    sql_latih = generate_create_table_sql(df_latih, 'data_latih')
    print(sql_latih)
    create_statements.append(("-- Table: data_latih", sql_latih))

if 'df_uji_y' in globals():
    print("\n2. CREATE TABLE for data_uji_y:")
    sql_uji_y = generate_create_table_sql(df_uji_y, 'data_uji_y')
    print(sql_uji_y)
    create_statements.append(("-- Table: data_uji_y", sql_uji_y))

## 5. Write Schema to SQL File

Menulis pernyataan CREATE TABLE yang telah dibuat ke file .sql untuk digunakan kemudian.

In [ ]:
# Write schema to SQL file
schema_filename = 'mysql_schema.sql'

try:
    with open(schema_filename, 'w', encoding='utf-8') as f:
        # Write header
        f.write("-- MySQL Schema Generated from Excel Files\n")
        f.write(f"-- Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("-- Source files: data_latih.xlsx, data_uji_y.xlsx\n\n")
        
        # Write database creation (optional)
        f.write("-- Create database (optional)\n")
        f.write("-- CREATE DATABASE stunting_db;\n")
        f.write("-- USE stunting_db;\n\n")
        
        # Write CREATE TABLE statements
        for comment, sql in create_statements:
            f.write(f"{comment}\n")
            f.write(f"{sql}\n")
    
    print(f"✓ Schema berhasil ditulis ke file: {schema_filename}")
    print(f"✓ File location: {os.path.abspath(schema_filename)}")
    
    # Display file content
    print(f"\n" + "="*60)
    print("CONTENT OF GENERATED SQL FILE:")
    print("="*60)
    with open(schema_filename, 'r', encoding='utf-8') as f:
        print(f.read())
        
except Exception as e:
    print(f"❌ Error writing schema file: {e}")